[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2026/blob/main/Individual-Contest/4_Double_Agent_Dilemma/code/grading/evaluate.ipynb)

# Double Agent Dilemma — evaluate a solution on Google Colab

Runs a solution the way the contest graded it, against the hidden leaderboard splits (now public in the [dataset](https://huggingface.co/datasets/IOAI-official/ioai-2026-double-agent-dilemma)).

**Evaluating your own notebook:** upload it via the Files pane as `/content/solution.ipynb`, then Run all. Without an upload, the stock baseline is evaluated.

> **Unofficial educational version** — provided so the task can be used outside the contest environment, reading the data directly from the Hugging Face dataset. The official contest artifacts are preserved in `code/baseline-original/` and `code/grading-original/`.

In [ ]:
# ============================ Colab setup (added) ============================
# Downloads the task dataset from Hugging Face and lays it out exactly as the
# contest environment did. Everything below this cell is the original baseline.
import os, sys, shutil, subprocess
from pathlib import Path
def sh(c): print('+',c); subprocess.run(c, shell=True, check=True)
sh('pip -q install timm safetensors huggingface_hub')
import json

from huggingface_hub import snapshot_download
DATA = Path(snapshot_download("IOAI-official/ioai-2026-double-agent-dilemma", repo_type="dataset"))
def link(src, dst):
    dst = Path(dst); dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_symlink() or dst.exists(): return
    os.symlink(src, dst)
# merged dataset root: split folders + *_answers files, public and private together
DSROOT = Path("/content/dsroot").resolve()
for sub in ("public","private"):
    d = DATA/sub
    if d.is_dir():
        for child in d.iterdir():
            link(child, DSROOT/child.name)
print("dataset root:", DSROOT, "->", sorted(p.name for p in DSROOT.iterdir()))


In [ ]:
EVAL_SPLIT = "test_leaderboard_a"   # or "test_leaderboard_b"

REPO = Path("/content/IOAI-2026")
if not REPO.exists(): sh(f"git clone --depth 1 https://github.com/IOAI-official/IOAI-2026 {REPO}")
WORK = Path("/content/work_dad"); WORK.mkdir(exist_ok=True)

SOL = Path("/content/solution.ipynb")
if not SOL.exists(): SOL = REPO/"Individual-Contest/4_Double_Agent_Dilemma/code/baseline/solution.ipynb"
shutil.copy(SOL, WORK/"solution.ipynb"); print("evaluating:", SOL, "on", EVAL_SPLIT)

# The baseline embeds the official scorer (compute_score mirrors evaluate.py):
# point TEST_SPLIT at the graded split and its own final cells print the score.
env = dict(os.environ, DATA_DIR=str(DSROOT),
           TRAIN_SPLIT="train", TEST_SPLIT=EVAL_SPLIT)
subprocess.run(f"cd {WORK} && jupyter nbconvert --to notebook --execute --output executed.ipynb "
               f"--ExecutePreprocessor.timeout=-1 solution.ipynb", shell=True, check=True, env=env)
nb = json.loads((WORK/"executed.ipynb").read_text())
for c in nb["cells"]:
    if c["cell_type"]=="code" and any(k in "".join(c["source"]) for k in ("compute_score","make_submission_zip")):
        for o in c.get("outputs",[]):
            print("".join(o.get("text",[])) if o.get("text") else "")
